In [1]:
# ── CELL 0: Environment Setup ─────────────────────────────────────────────
import subprocess, sys

print("Pinning NumPy < 2 ...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "numpy==1.26.4", "--force-reinstall"], check=True)
print("✅ NumPy 1.26.4 pinned")
import os

!python -c "import torch; print('✅ PyTorch', torch.__version__, '| GPUs:', torch.cuda.device_count(), '| GPU0:', torch.cuda.get_device_name(0))"
!python -c "import numpy as np; print('✅ NumPy', np.__version__)"

!pip -q install \
    "numpy==1.26.4" "transformers==4.40.2" "tokenizers==0.19.1" \
    "diffusers==0.26.3" "accelerate==0.29.3" "huggingface_hub" \
    "opencv-python==4.9.0.80" "opencv-contrib-python==4.9.0.80" \
    "onnxruntime-gpu==1.18.0" "onnx==1.16.1" "tyro==0.8.5" \
    "lmdb==1.4.1" "pykalman==0.9.7" "albumentations==1.4.10" \
    "scipy==1.13.1" "scikit-image==0.22.0" "scikit-learn==1.4.2" \
    "mediapipe" "imageio==2.33.0" "imageio-ffmpeg==0.4.9" \
    "ffmpeg-python==0.2.0" "einops==0.7.0" "omegaconf==2.3.0" \
    "tqdm" "rich" "pillow>=10.2.0" "gradio>=4.20.0"

!pip -q install xformers==0.0.29.post3
print("✅ Cell 0 done — environment ready")


Pinning NumPy < 2 ...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 79.9 MB/s eta 0:00:00
✅ NumPy 1.26.4 pinned


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
kaggle-environments 1.27.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python 4.12.0.88 requires numpy<

✅ PyTorch 2.9.0+cu126 | GPUs: 2 | GPU0: Tesla T4
✅ NumPy 1.26.4
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.0/138.0 kB 3.6 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 881.5/881.5 kB 21.8 MB/s eta 0:00:0000:01
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 94.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 101.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 83.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 297.6/297.6 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.2/62.2 MB 32.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.3/68.3 MB 28.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.8/199.8 MB 9.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [16]:
# ── CELL 1: Models + Pipeline Functions ───────────────────────────────────
import os, subprocess, glob, json, shutil, math, ctypes, random, threading, time
import cv2, numpy as np
from PIL import Image, ImageFilter
from huggingface_hub import snapshot_download

WORK       = "/kaggle/working"
LP_DIR     = "/kaggle/working/LivePortrait"
ESRGAN_PTH = "/kaggle/working/RealESRGAN_x2plus.pth"

if not os.path.exists(LP_DIR):
    subprocess.run(["git","clone","https://github.com/KwaiVGI/LivePortrait",LP_DIR],check=True)
    print("✅ LivePortrait cloned")
else: print("✅ LivePortrait present")

if not os.path.exists(os.path.join(LP_DIR,"pretrained_weights","liveportrait")):
    os.chdir(LP_DIR)
    snapshot_download(repo_id="KwaiVGI/LivePortrait",local_dir="pretrained_weights",
                      ignore_patterns=["*.git*","README.md","docs/*"])
    print("✅ LivePortrait weights downloaded")
else: print("✅ LivePortrait weights present")

if not os.path.exists(ESRGAN_PTH):
    subprocess.run(["wget","-q","-O",ESRGAN_PTH,
        "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x2plus.pth"],check=True)
    print("✅ ESRGAN weights downloaded")
else: print("✅ ESRGAN weights present")

os.chdir(WORK)

_DT={1:'uint8',2:'uint16',4:'float32',8:'float64'}
def cv2np(a,t='float32'):
    d=_DT[int(a.itemsize)];s=tuple(int(x) for x in a.shape);n=int(a.nbytes)
    r=np.empty(s,dtype=d)
    ctypes.memmove(r.ctypes.data,ctypes.c_void_p(a.__array_interface__['data'][0]),n)
    return r.astype(t) if t!=d else r
def np2cv(a,c):
    ctypes.memmove(c.__array_interface__['data'][0],a.ctypes.data,int(c.nbytes)); return c
def ffprobe_info(path):
    r=subprocess.run(["ffprobe","-v","quiet","-print_format","json","-show_streams",path],
                     capture_output=True,text=True)
    s=json.loads(r.stdout).get("streams",[])
    return next((x for x in s if x["codec_type"]=="video"),{})

def clear_gpu_memory(log=print):
    log("🧹 Clearing GPU VRAM ...")
    r=subprocess.run(["python","-c","""
import torch
n=torch.cuda.device_count()
for i in range(n):
    with torch.cuda.device(i):
        torch.cuda.empty_cache(); torch.cuda.ipc_collect()
    print(f'  GPU {i}: {torch.cuda.memory_allocated(i)/1e9:.2f} GB remaining')
print(f'Cleared {n} GPU(s)')
"""],capture_output=True,text=True)
    for line in r.stdout.strip().split("\n"): log(f"   {line}")
    log("✅ GPU VRAM cleared")

def normalize_video(source_image_path,driving_video_path,log=print):
    log("📥 Normalising inputs ...")
    src=f"{WORK}/my_image.png"; trm=f"{WORK}/driving_trimmed.mp4"
    shutil.copy(source_image_path,src)
    subprocess.run(["ffmpeg","-y","-i",driving_video_path,
        "-vf","scale=512:768:force_original_aspect_ratio=decrease,"
              "pad=512:768:(ow-iw)/2:(oh-ih)/2:color=black",
        "-r","25","-c:v","libx264","-crf","18","-preset","fast",trm],
        check=True,stderr=subprocess.DEVNULL)
    r=subprocess.run(["ffprobe","-v","quiet","-select_streams","v:0",
                       "-count_packets","-show_entries","stream=nb_read_packets",
                       "-of","csv=p=0",trm],capture_output=True,text=True)
    log(f"✅ Driving video ready — {r.stdout.strip()} frames @ 512×768 25fps")
    return src,trm

def run_liveportrait(source_image,trimmed_video,motion_multiplier=0.65,log=print):
    log(f"🎭 [GPU 0] Running LivePortrait (motion×{motion_multiplier}) ...")
    env=os.environ.copy(); env["CUDA_VISIBLE_DEVICES"]="0"
    _done=f"{LP_DIR}/.lip_zero_patched"
    if not os.path.exists(_done):
        for cfg in [f"{LP_DIR}/src/config/argument_config.py",
                    f"{LP_DIR}/src/config/inference_config.py"]:
            if os.path.exists(cfg):
                with open(cfg,"r") as fh: s=fh.read()
                p=s.replace("flag_lip_zero: bool = True","flag_lip_zero: bool = False"
                   ).replace("flag_lip_zero=True","flag_lip_zero=False")
                if p!=s:
                    with open(cfg,"w") as fh: fh.write(p)
                    log(f"   ✅ Patched flag_lip_zero→False in {os.path.basename(cfg)}")
        open(_done,"w").close()
    os.chdir(LP_DIR)
    subprocess.run(["python","inference.py","-s",source_image,"-d",trimmed_video,
        "--flag-relative-motion","--flag-pasteback","--flag-do-crop",
        "--driving-multiplier",str(motion_multiplier),"--scale","3.0"],
        check=True,stderr=subprocess.DEVNULL,env=env)
    os.chdir(WORK)
    vids=[v for v in glob.glob(f"{LP_DIR}/animations/**/*.mp4",recursive=True)
          if 'concat' not in os.path.basename(v)]
    lp=max(vids if vids else glob.glob(f"{LP_DIR}/animations/**/*.mp4",recursive=True),
           key=os.path.getmtime)
    log(f"✅ Face animation → {os.path.basename(lp)}")
    return lp

# ── NEW: Hair Motion Tracker ──────────────────────────────────────────────
# ── NEW: Hair Motion Tracker (v2) ─────────────────────────────────────────
def apply_hair_motion(lp_video, source_image, log=print):
    log("💇 [HAIR] Applying hair motion tracking (v3 — incremental) ...")
    OUT = f"{WORK}/lp_hair_motion.mp4"
    bi  = ffprobe_info(lp_video)
    if not bi: log("   ⚠️ [HAIR] ffprobe failed — SKIPPED"); return lp_video

    VW  = (int(bi["width"])  // 2) * 2
    VH  = (int(bi["height"]) // 2) * 2
    num, den = bi.get("r_frame_rate", "25/1").split("/")
    FPS = float(num) / float(den)

    src_bgr  = cv2.resize(cv2.imread(source_image), (VW, VH))
    src_gray = cv2.cvtColor(src_bgr, cv2.COLOR_BGR2GRAY)

    fc = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    faces = None
    for sf, mn, msz in [(1.05,6,(60,60)),(1.05,4,(50,50)),(1.1,3,(40,40))]:
        det = fc.detectMultiScale(src_gray, sf, mn, minSize=msz)
        if len(det) > 0: faces = det; break
    if faces is None: log("   ⚠️ [HAIR] No face detected — SKIPPED"); return lp_video

    fx, fy, fw, fh = max(faces, key=lambda r: r[2]*r[3])
    chin_y = fy + fh
    log(f"   [HAIR] Face ({fx},{fy},{fw},{fh}) | chin={chin_y} | {VW}×{VH}")

    # ── Hair mask: top-of-frame → below chin, wide lateral pads ──────────
    pad_x       = int(fw * 0.60)
    hair_bottom = min(VH, chin_y + int(fh * 0.35))
    x_l = max(0, fx - pad_x);  x_r = min(VW, fx + fw + pad_x)

    hm = np.zeros((VH, VW), dtype=np.float32)
    hm[0:hair_bottom, x_l:x_r] = 1.0
    # punch out inner face so LP animation is preserved
    hm[max(0,fy+int(fh*0.10)):min(VH,chin_y-int(fh*0.06)),
       max(0,fx+int(fw*0.12)):min(VW,fx+fw-int(fw*0.12))] = 0.0
    # fade top edge
    fade_top = min(30, fy)
    if fade_top > 0:
        hm[0:fade_top, x_l:x_r] *= np.linspace(0, 1, fade_top)[:, np.newaxis]
    # fade bottom edge
    bot_start = max(0, hair_bottom - 40)
    n_rows = hair_bottom - bot_start
    if n_rows > 1:
        hm[bot_start:hair_bottom, x_l:x_r] *= np.linspace(1, 0, n_rows)[:, np.newaxis]

    hm  = cv2.GaussianBlur(hm, (71, 71), 32)
    hm3 = np.stack([hm]*3, axis=-1);  inv3 = 1.0 - hm3

    # ── Seed feature points from source image ────────────────────────────
    ty1 = max(0, fy - int(fh*0.20));  ty2 = chin_y
    tx1 = max(0, fx - int(fw*0.05));  tx2 = min(VW, fx + fw + int(fw*0.05))
    seed = cv2.goodFeaturesToTrack(src_gray[ty1:ty2, tx1:tx2],
                                    maxCorners=150, qualityLevel=0.004,
                                    minDistance=4, blockSize=7)
    if seed is None or len(seed) < 6:
        log("   ⚠️ [HAIR] Too few seed features — SKIPPED"); return lp_video
    seed[:, 0, 0] += tx1;  seed[:, 0, 1] += ty1

    lk = dict(winSize=(31,31), maxLevel=5,
               criteria=(cv2.TERM_CRITERIA_EPS|cv2.TERM_CRITERIA_COUNT, 30, 0.005))

    rdr = subprocess.Popen(["ffmpeg","-i",lp_video,"-f","rawvideo","-pix_fmt","bgr24",
                             "-vf",f"scale={VW}:{VH}","-"],
                            stdout=subprocess.PIPE, stderr=subprocess.DEVNULL)
    wtr = subprocess.Popen(["ffmpeg","-y","-f","rawvideo","-pix_fmt","bgr24",
                             "-s",f"{VW}x{VH}","-r",str(FPS),"-i","pipe:0",
                             "-c:v","libx264","-crf","18","-preset","fast",
                             "-pix_fmt","yuv420p",OUT],
                            stdin=subprocess.PIPE, stderr=subprocess.PIPE)

    fsz   = VW * VH * 3
    M_acc = np.eye(2, 3, dtype=np.float64)   # identity — accumulate head motion here
    prev_gray = src_gray.copy()
    cur_pts   = seed.copy()
    n = 0;  ok = 0

    try:
        while True:
            raw = rdr.stdout.read(fsz)
            if len(raw) < fsz: break
            frm = np.frombuffer(raw, dtype=np.uint8).reshape((VH,VW,3)).copy()
            fg  = cv2.cvtColor(frm, cv2.COLOR_BGR2GRAY)

            p2, st, _ = cv2.calcOpticalFlowPyrLK(prev_gray, fg, cur_pts, None, **lk)
            gs = cur_pts[st[:,0]==1].reshape(-1,1,2)
            gd = p2[st[:,0]==1].reshape(-1,1,2)

            if len(gs) >= 6:
                M_inc, _ = cv2.estimateAffinePartial2D(gs, gd, method=cv2.RANSAC,
                                                        ransacReprojThreshold=1.5,
                                                        confidence=0.99)
                if M_inc is not None:
                    # compose: accumulated = incremental @ accumulated
                    M_acc = (np.vstack([M_inc,[0,0,1]]) @
                             np.vstack([M_acc,[0,0,1]]))[:2]
                    warped = cv2.warpAffine(src_bgr, M_acc, (VW,VH),
                                            flags=cv2.INTER_LINEAR,
                                            borderMode=cv2.BORDER_REFLECT_101)
                    frm = np.clip(frm.astype('float32')*inv3 +
                                  warped.astype('float32')*hm3, 0, 255).astype('uint8')
                    ok += 1
                cur_pts = gd  # advance tracked points
            else:
                # reseed from current frame when points are lost
                log(f"   [HAIR] frame {n:04d}: reseed ({len(gs)} pts left)")
                rp = cv2.goodFeaturesToTrack(fg[ty1:ty2,tx1:tx2],
                                              maxCorners=150, qualityLevel=0.004,
                                              minDistance=4, blockSize=7)
                if rp is not None and len(rp) >= 6:
                    rp[:,0,0] += tx1;  rp[:,0,1] += ty1;  cur_pts = rp

            wtr.stdin.write(frm.tobytes())
            prev_gray = fg
            if n % 25 == 0: log(f"   [HAIR] frame {n:04d}  tracked={ok}")
            n += 1

    except BrokenPipeError:
        raise RuntimeError(f"[HAIR] writer crashed:\n{wtr.stderr.read().decode(errors='replace')}")

    rdr.stdout.close(); rdr.wait()
    wtr.stdin.close()
    werr = wtr.stderr.read().decode(errors='replace');  wtr.wait()
    if wtr.returncode != 0:
        raise RuntimeError(f"[HAIR] ffmpeg exited {wtr.returncode}:\n{werr}")
    log(f"✅ [HAIR] Done — {n} frames | head-tracked: {ok}/{n}")
    return OUT


def amplify_lip_opening(lp_video, source_image, lip_gain=2.8, log=print):
    log(f"👄 [LIP-AMP] Starting  lip_gain={lip_gain}")
    OUT = f"{WORK}/lp_lips_boosted.mp4"
    bi = ffprobe_info(lp_video)
    if not bi: log("   ⚠️ [LIP-AMP] ffprobe returned nothing — SKIPPED"); return lp_video

    VW  = (int(bi["width"])//2)*2;  VH = (int(bi["height"])//2)*2
    num, den = bi.get("r_frame_rate","25/1").split("/");  FPS = float(num)/float(den)
    log(f"   [LIP-AMP] Video: {VW}×{VH} @ {FPS:.2f}fps")

    src_bgr  = cv2.resize(cv2.imread(source_image), (VW, VH))
    gray_src = cv2.cvtColor(src_bgr, cv2.COLOR_BGR2GRAY)

    fc = cv2.CascadeClassifier(cv2.data.haarcascades+'haarcascade_frontalface_default.xml')
    faces = None
    for sf, mn, msz in [(1.05,6,(60,60)),(1.05,4,(50,50)),(1.1,3,(40,40))]:
        det = fc.detectMultiScale(gray_src, sf, mn, minSize=msz)
        if len(det) > 0: faces = det; break
    if faces is None: log("   ⚠️ [LIP-AMP] No face detected — SKIPPED"); return lp_video

    fx, fy, fw, fh = max(faces, key=lambda r: r[2]*r[3])
    my1 = max(0,    int(fy + fh*0.65));  my2 = min(VH-1, int(fy + fh*0.93))
    mx1 = max(0,    int(fx + fw*0.18));  mx2 = min(VW-1, int(fx + fw*0.82))
    log(f"   [LIP-AMP] Face=({fx},{fy},{fw},{fh})  LipROI x=[{mx1}:{mx2}] y=[{my1}:{my2}]")

    raw_mask = np.zeros((VH, VW), dtype='float32')
    raw_mask[my1:my2, mx1:mx2] = 1.0
    ksize  = max(3, (((my2-my1)//4)|1));  blur_k = ksize*6+1
    mask   = cv2.GaussianBlur(raw_mask, (blur_k, blur_k), ksize)
    mask3  = np.stack([mask]*3, axis=-1);  inv3 = 1.0 - mask3

    # ── SOURCE IMAGE is the "mouth at rest" reference — NOT frame 0 ───────
    src_f32 = src_bgr.astype('float32')

    rdr = subprocess.Popen(["ffmpeg","-i",lp_video,"-f","rawvideo","-pix_fmt","bgr24",
                             "-vf",f"scale={VW}:{VH}","-"],
                            stdout=subprocess.PIPE, stderr=subprocess.DEVNULL)
    wtr = subprocess.Popen(["ffmpeg","-y","-f","rawvideo","-pix_fmt","bgr24",
                             "-s",f"{VW}x{VH}","-r",str(FPS),"-i","pipe:0",
                             "-c:v","libx264","-crf","18","-preset","fast",
                             "-pix_fmt","yuv420p",OUT],
                            stdin=subprocess.PIPE, stderr=subprocess.PIPE)

    fsz = VW*VH*3;  n = 0;  mx_d = 0.0;  sm_d = 0.0

    try:
        while True:
            raw = rdr.stdout.read(fsz)
            if len(raw) < fsz: break
            frm = np.frombuffer(raw, dtype=np.uint8).reshape((VH,VW,3)).astype('float32')

            # delta = what LP added relative to the still source photo
            delta    = frm - src_f32
            lip_d    = np.abs(delta[my1:my2, mx1:mx2])
            md = float(lip_d.mean());  xd = float(lip_d.max())
            sm_d += md
            if xd > mx_d: mx_d = xd

            # amplify: push lip pixels further along LP's direction
            amplified = src_f32 + delta * lip_gain
            res = frm * inv3 + np.clip(amplified, 0, 255) * mask3

            wtr.stdin.write(np.clip(res, 0, 255).astype('uint8').tobytes())
            if n % 25 == 0: log(f"   [LIP-AMP] frame {n:04d}  Δ mean={md:.2f} max={xd:.2f}")
            n += 1

    except BrokenPipeError:
        raise RuntimeError(f"[LIP-AMP] writer crashed:\n{wtr.stderr.read().decode(errors='replace')}")

    rdr.stdout.close();  rdr.wait()
    wtr.stdin.close();   werr = wtr.stderr.read().decode(errors='replace');  wtr.wait()
    if wtr.returncode != 0:
        raise RuntimeError(f"[LIP-AMP] exited {wtr.returncode}:\n{werr}")

    avg = sm_d / max(n, 1)
    log(f"✅ [LIP-AMP] Done — {n} frames  avg_Δ={avg:.2f}  peak_Δ={mx_d:.2f}")
    if avg < 1.0: log("   ⚠️ [LIP-AMP] avg_Δ<1 — LP produced very little lip motion.")
    else:         log(f"   ✅ [LIP-AMP] Good lip motion — amplified ×{lip_gain}")
    return OUT

def run_body_animation(source_image,trimmed_video,log=print):
    log("🕺 [CPU] Body animation ...")
    BODY_VIDEO=f"{WORK}/body_animation.mp4"; RAW_FRAMES=f"{WORK}/body_frames.raw"
    src=cv2.imread(source_image); H,W=src.shape[:2]
    H=H if H%2==0 else H-1; W=W if W%2==0 else W-1; src=src[:H,:W]
    src_f32=cv2np(src,'float32')
    gray=cv2.cvtColor(src,cv2.COLOR_BGR2GRAY)
    map1_tpl=cv2.normalize(gray,None,0,1,cv2.NORM_MINMAX,dtype=cv2.CV_32F)
    map2_tpl=cv2.normalize(gray,None,0,1,cv2.NORM_MINMAX,dtype=cv2.CV_32F)
    fc=cv2.CascadeClassifier(cv2.data.haarcascades+'haarcascade_frontalface_default.xml')
    faces=fc.detectMultiScale(gray,1.1,5,minSize=(60,60))
    if len(faces)>0:
        fx,fy,fw,fh=max(faces,key=lambda r:r[2]*r[3])
        NECK_Y=min(int(fy+fh*1.05),H-1); SHOULDER_Y=min(NECK_Y+50,H-1)
    else: NECK_Y=H//3; SHOULDER_Y=NECK_Y+50
    log(f"   Neck y={NECK_Y}, shoulder y={SHOULDER_Y}")
    cap=cv2.VideoCapture(trimmed_video)
    FPS=float(cap.get(cv2.CAP_PROP_FPS) or 25.0); N=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    BREATH_AMP_VERT=4.2; BREATH_AMP_HOR=3.8; BREATH_HZ=0.385
    SHOULDER_AMP_VERT=2.8; SHOULDER_HZ=0.385; SWAY_AMP_HOR=1.85; SWAY_HZ=0.155
    SPEECH_AMP_HOR=1.0; SPEECH_HZ=1.8; RANDOM_AMP=0.3; RAMP_FRAMES=20
    weight_vert=np.zeros(H,dtype='float32')
    chest_center=int(SHOULDER_Y+(H-SHOULDER_Y)*0.3)
    for y in range(SHOULDER_Y,H):
        weight_vert[y]=((y-SHOULDER_Y)/(chest_center-SHOULDER_Y) if y<chest_center
                        else max((1.0-(y-chest_center)/(H-chest_center)*0.5),0.5))
    x_coords=np.arange(W); cx=W//2; sigma=0.12*W
    horiz=np.exp(-((x_coords-cx)**2)/(2*sigma**2))
    torso=weight_vert[:,np.newaxis]*horiz[np.newaxis,:]
    hsign=(cx-x_coords)/(W//2); dx_bw=torso*hsign[np.newaxis,:]
    sv=np.zeros(H,dtype='float32'); spk=SHOULDER_Y+20; sfd=60
    for y in range(SHOULDER_Y,H):
        sv[y]=1.0 if y<spk else max(0.0,1.0-(y-spk)/sfd)
    sw=sv[:,np.newaxis]*horiz[np.newaxis,:]
    ys_raw,xs_raw=np.mgrid[0:H,0:W].astype('float32')
    mr=np.zeros((H,1),dtype='float32'); fade=30
    for y in range(H):
        if y<NECK_Y-fade: mr[y]=0.0
        elif y<NECK_Y+fade: mr[y]=(y-(NECK_Y-fade))/(2*fade)
        else: mr[y]=1.0
    m3=np.stack([mr*np.ones((1,W),'float32')]*3,axis=-1); m3i=1.0-m3
    eh=np.clip(np.minimum(np.linspace(0,1,W),np.linspace(1,0,W))*6,0,1).astype('float32')
    ev=np.clip(np.minimum(np.linspace(0,1,H),np.linspace(1,0,H))*6,0,1).astype('float32')
    edge_mask=ev[:,np.newaxis]*eh[np.newaxis,:]
    pr_dx=pr_dy=0.0; rs=0.7
    with open(RAW_FRAMES,'wb') as raw_f:
        for i in range(N):
            ramp=min(1.0,i/RAMP_FRAMES); t=i/FPS
            bv=math.sin(2*math.pi*BREATH_HZ*t); shv=math.sin(2*math.pi*SHOULDER_HZ*t+0.3)
            sv2=math.sin(2*math.pi*SWAY_HZ*t+0.7); sh=math.cos(2*math.pi*SPEECH_HZ*t)
            spv=math.sin(2*math.pi*SPEECH_HZ*t+0.7)
            pr_dx=rs*pr_dx+(1-rs)*(random.uniform(-1,1)*RANDOM_AMP)
            pr_dy=rs*pr_dy+(1-rs)*(random.uniform(-1,1)*RANDOM_AMP)
            ddx=(dx_bw*(bv*BREATH_AMP_HOR*ramp)+torso*(sv2*SWAY_AMP_HOR*ramp)
                 +torso*0.3*(sh*SPEECH_AMP_HOR*ramp)+torso*pr_dx)
            ddy=(torso*(bv*BREATH_AMP_VERT*ramp)+sw*(shv*SHOULDER_AMP_VERT*ramp)
                 +torso*0.3*(spv*0.5*ramp)+torso*pr_dy)
            dx=xs_raw+ddx*edge_mask; dy=ys_raw+ddy*edge_mask
            np2cv(dx.astype('float32'),map1_tpl); np2cv(dy.astype('float32'),map2_tpl)
            warped=cv2np(cv2.remap(src,map1_tpl,map2_tpl,
                                   cv2.INTER_LINEAR,cv2.BORDER_REFLECT_101),'float32')
            raw_f.write(np.clip(src_f32*m3i+warped*m3,0,255).astype('uint8').tobytes())
            if i%50==0: log(f"   Body frame {i}/{N}")
    subprocess.run(["ffmpeg","-y","-f","rawvideo","-pix_fmt","bgr24",
                    "-s",f"{W}x{H}","-r",str(FPS),"-i",RAW_FRAMES,
                    "-c:v","libx264","-crf","18","-preset","ultrafast",
                    "-pix_fmt","yuv420p",BODY_VIDEO],check=True,stderr=subprocess.DEVNULL)
    os.remove(RAW_FRAMES)
    log(f"✅ Body animation → {os.path.getsize(BODY_VIDEO)/1e6:.1f} MB")
    return BODY_VIDEO

def composite_videos(lp_video,body_video,source_image,trimmed_video,log=print):
    log("🎬 Compositing face + body ...")
    FINAL=f"{WORK}/final_output.mp4"
    bi=ffprobe_info(body_video)
    BW=int(bi["width"]); BH=int(bi["height"])
    num,den=bi.get("r_frame_rate","25/1").split("/"); FPS_C=float(num)/float(den)
    src_r=cv2.resize(cv2.imread(source_image),(BW,BH))
    gray=cv2.cvtColor(src_r,cv2.COLOR_BGR2GRAY)
    fc=cv2.CascadeClassifier(cv2.data.haarcascades+'haarcascade_frontalface_default.xml')
    fx=fy=fw2=fh=None
    for sf,mn,msz in [(1.05,6,(60,60)),(1.05,4,(50,50)),(1.1,3,(40,40))]:
        fcs=fc.detectMultiScale(gray,scaleFactor=sf,minNeighbors=mn,minSize=msz)
        if len(fcs)>0: fx,fy,fw2,fh=max(fcs,key=lambda r:r[2]*r[3]); break
    if fx is None: raise RuntimeError("❌ No face detected in source image")
    raw_neck=int(fy+fh*1.40); min_below_chin=int(fy+fh)+20
    NECK_Y=min(max(raw_neck,min_below_chin),BH-1)
    log(f"   Face: y={fy} h={fh} → chin={fy+fh} → neck split at y={NECK_Y}")
    FADE=20; lp_mask=np.zeros((BH,BW),dtype=np.float32)
    for y in range(BH):
        if y<NECK_Y-FADE: lp_mask[y,:]=1.0
        elif y<NECK_Y+FADE: lp_mask[y,:]=(NECK_Y+FADE-y)/(2.0*FADE)
        else: lp_mask[y,:]=0.0
    lp_m3=np.stack([lp_mask]*3,axis=-1); body_m3=1.0-lp_m3
    def opipe(p,W,H):
        return subprocess.Popen(["ffmpeg","-i",p,"-f","rawvideo","-pix_fmt","bgr24",
                                  "-vf",f"scale={W}:{H}","-"],
                                 stdout=subprocess.PIPE,stderr=subprocess.DEVNULL)
    def rframe(pipe,W,H):
        raw=pipe.stdout.read(W*H*3)
        if len(raw)<W*H*3: return None
        return np.frombuffer(raw,dtype=np.uint8).reshape((H,W,3)).copy()
    fp=opipe(lp_video,BW,BH); bp=opipe(body_video,BW,BH)
    fout=subprocess.Popen(["ffmpeg","-y","-f","rawvideo","-pix_fmt","bgr24",
                            "-s",f"{BW}x{BH}","-r",str(FPS_C),"-i","pipe:0",
                            "-i",trimmed_video,"-map","0:v","-map","1:a?",
                            "-c:v","libx264","-crf","18","-preset","fast",
                            "-pix_fmt","yuv420p","-shortest",FINAL],
                           stdin=subprocess.PIPE,stderr=subprocess.DEVNULL)
    idx=0
    while True:
        fb=rframe(fp,BW,BH); bb=rframe(bp,BW,BH)
        if fb is None or bb is None: break
        comp=fb.astype('float32')*lp_m3+bb.astype('float32')*body_m3
        fout.stdin.write(np.clip(comp,0,255).astype('uint8').tobytes()); idx+=1
    fp.stdout.close(); fp.wait(); bp.stdout.close(); bp.wait()
    fout.stdin.close(); fout.wait()
    log(f"✅ Composited {idx} frames → {os.path.getsize(FINAL)/1e6:.1f} MB")
    return FINAL

def auto_batch_size(final_video,target_gb=12.0,log=print):
    info=ffprobe_info(final_video)
    W=int(info.get("width",480)); H=int(info.get("height",640))
    pw=(int(W*3/8))*2; ph=(int(H*3/8))*2
    pixels=pw*ph; K=1.69e-6; MODEL_GB=0.5
    vram_per_frame=max(K*pixels,0.05)
    batch=min(max(1,int((target_gb-MODEL_GB)/vram_per_frame)),48)
    log(f"   Auto VRAM tune: {W}×{H} → {pw}×{ph} | batch={batch} → ~{MODEL_GB+batch*vram_per_frame:.1f}GB/GPU")
    return batch

_WORKER="""
import sys,os,json,cv2,numpy as np,torch,torch.nn as nn,torch.nn.functional as F
class ResidualDenseBlock(nn.Module):
    def __init__(self,nf=64,gc=32):
        super().__init__()
        self.conv1=nn.Conv2d(nf,gc,3,1,1);self.conv2=nn.Conv2d(nf+gc,gc,3,1,1)
        self.conv3=nn.Conv2d(nf+2*gc,gc,3,1,1);self.conv4=nn.Conv2d(nf+3*gc,gc,3,1,1)
        self.conv5=nn.Conv2d(nf+4*gc,nf,3,1,1);self.lrelu=nn.LeakyReLU(0.2,True)
    def forward(self,x):
        x1=self.lrelu(self.conv1(x));x2=self.lrelu(self.conv2(torch.cat((x,x1),1)))
        x3=self.lrelu(self.conv3(torch.cat((x,x1,x2),1)));x4=self.lrelu(self.conv4(torch.cat((x,x1,x2,x3),1)))
        return self.conv5(torch.cat((x,x1,x2,x3,x4),1))*0.2+x
class RRDB(nn.Module):
    def __init__(self,nf,gc=32):
        super().__init__()
        self.rdb1=ResidualDenseBlock(nf,gc);self.rdb2=ResidualDenseBlock(nf,gc);self.rdb3=ResidualDenseBlock(nf,gc)
    def forward(self,x): return self.rdb3(self.rdb2(self.rdb1(x)))*0.2+x
class RRDBNet(nn.Module):
    def __init__(self,ic=3,oc=3,nf=64,nb=23,gc=32,scale=2):
        super().__init__()
        self.scale=scale
        self.conv_first=nn.Conv2d(ic*4 if scale==2 else ic,nf,3,1,1)
        self.body=nn.Sequential(*[RRDB(nf,gc) for _ in range(nb)])
        self.conv_body=nn.Conv2d(nf,nf,3,1,1);self.conv_up1=nn.Conv2d(nf,nf,3,1,1)
        self.conv_up2=nn.Conv2d(nf,nf,3,1,1);self.conv_hr=nn.Conv2d(nf,nf,3,1,1)
        self.conv_last=nn.Conv2d(nf,oc,3,1,1);self.lrelu=nn.LeakyReLU(0.2,True)
    def forward(self,x):
        f=F.pixel_unshuffle(x,2) if self.scale==2 else x
        f=self.conv_first(f);f=f+self.conv_body(self.body(f))
        f=self.lrelu(self.conv_up1(F.interpolate(f,scale_factor=2,mode='nearest')))
        f=self.lrelu(self.conv_up2(F.interpolate(f,scale_factor=2,mode='nearest')))
        return self.conv_last(self.lrelu(self.conv_hr(f)))
gpu_id=int(sys.argv[1]);fps=json.loads(sys.argv[2])
fo=sys.argv[3];bs=int(sys.argv[4]);wt=sys.argv[5]
dev=torch.device('cuda:0')
os.makedirs(fo,exist_ok=True)
m=RRDBNet();ck=torch.load(wt,map_location='cpu')
m.load_state_dict(ck.get('params_ema',ck.get('params',ck)),strict=True)
m.eval().half().to(dev)
def pre(paths):
    imgs=[cv2.cvtColor(cv2.imread(p),cv2.COLOR_BGR2RGB) for p in paths]
    t=[(torch.from_numpy(i).float()/255.0).permute(2,0,1).unsqueeze(0) for i in imgs]
    return torch.cat(t,0).half().to(dev)
def post(t):
    o=t.float().clamp(0,1).cpu().numpy()
    return [cv2.cvtColor((o[i].transpose(1,2,0)*255).astype('uint8'),cv2.COLOR_RGB2BGR) for i in range(o.shape[0])]
total=len(fps)
torch.cuda.reset_peak_memory_stats(dev)
with torch.no_grad():
    for s in range(0,total,bs):
        bp=fps[s:s+bs];out=post(m(pre(bp)))
        for p,img in zip(bp,out): cv2.imwrite(f'{fo}/{os.path.basename(p)}',img)
        done=min(s+bs,total);pk=torch.cuda.max_memory_allocated(dev)/1e9
        print(f'[GPU {gpu_id}] {done}/{total} | VRAM {pk:.1f}GB',flush=True)
print(f'[GPU {gpu_id}] Done. Peak: {torch.cuda.max_memory_allocated(dev)/1e9:.2f}GB',flush=True)
"""
_WP="/tmp/esrgan_worker.py"
with open(_WP,"w") as f: f.write(_WORKER)

def run_esrgan(final_video,use_dual_gpu=True,log=print):
    log("🔍 Starting Real-ESRGAN 1.5× upscaling ...")
    FI=f"{WORK}/frames_in"; FO=f"{WORK}/frames_esr"; EV=f"{WORK}/enhanced_output.mp4"
    for d in [FI,FO]:
        if os.path.exists(d): shutil.rmtree(d)
        os.makedirs(d)
    subprocess.run(["ffmpeg","-y","-i",final_video,
                    "-vf","scale=trunc(iw*3/8)*2:trunc(ih*3/8)*2",
                    f"{FI}/frame_%05d.png"],check=True,stderr=subprocess.DEVNULL)
    all_frames=sorted(glob.glob(f"{FI}/frame_*.png")); total=len(all_frames)
    batch_size=auto_batch_size(final_video,target_gb=12.0,log=log)
    import torch; n_gpus=min(torch.cuda.device_count(),2) if use_dual_gpu else 1
    log(f"   {total} frames | {n_gpus} GPU(s) | batch={batch_size} | scale=1.5×")
    mid=total//2; splits=[all_frames[:mid],all_frames[mid:]] if n_gpus==2 else [all_frames]
    errors=[]
    def worker(gid,fl):
        env=os.environ.copy()
        env["CUDA_VISIBLE_DEVICES"]=str(gid)
        env["PYTORCH_CUDA_ALLOC_CONF"]="expandable_segments:True"
        res=subprocess.run(["python",_WP,str(gid),json.dumps(fl),FO,str(batch_size),ESRGAN_PTH],
                           capture_output=True,text=True,env=env)
        for line in res.stdout.strip().split("\n"): log(f"   {line}")
        if res.returncode!=0: errors.append(res.stderr[-600:])
    ts=[threading.Thread(target=worker,args=(i,splits[i])) for i in range(n_gpus)]
    t0=time.time()
    for t in ts: t.start()
    for t in ts: t.join()
    if errors: raise RuntimeError("ESRGAN failed:\n"+errors[0])
    log(f"✅ Upscaling done in {time.time()-t0:.1f}s")
    all_out=sorted(glob.glob(f"{FO}/frame_*.png"))
    for i,fp in enumerate(all_out): os.rename(fp,f"{FO}/seq_{i:05d}.png")
    subprocess.run(["ffmpeg","-y","-framerate","25","-i",f"{FO}/seq_%05d.png",
                    "-i",f"{WORK}/driving_trimmed.mp4","-map","0:v","-map","1:a?",
                    "-c:v","libx264","-crf","16","-preset","slow",
                    "-pix_fmt","yuv420p","-shortest",EV],check=True,stderr=subprocess.DEVNULL)
    log(f"✅ Enhanced → {os.path.getsize(EV)/1e6:.1f} MB")
    return EV

# ── Master Pipeline ───────────────────────────────────────────────────────
def full_pipeline(source_image_path,driving_video_path,
                  motion_multiplier=0.55,enable_upscale=True,
                  use_dual_gpu=True,log=print):
    log("━"*52)
    log("🚀 NeuroAnimate Pipeline Starting")
    log("━"*52)
    source,trimmed=normalize_video(source_image_path,driving_video_path,log)
    log("⚡ Stage 1: LivePortrait (GPU 0) ∥ Body Animation (CPU) — parallel ...")
    lp_r=[None]; bd_r=[None]; errs=[]

    def run_lp():
        try:
            raw_lp = run_liveportrait(source, trimmed, motion_multiplier, log)
            # ── NEW: make hair follow head motion ──────────────────────────
            try:
                lp_hair = apply_hair_motion(raw_lp, source, log=log)
            except Exception as hair_e:
                import traceback
                log(f"   ⚠️ Hair motion error (using raw LP):\n{traceback.format_exc()}")
                lp_hair = raw_lp
            # ── Lip amplifier ───────────────────────────────────────────────
            try:
                lp_r[0] = amplify_lip_opening(lp_hair, source, lip_gain=2.8, log=log)
            except Exception as lip_e:
                import traceback
                log(f"   ⚠️ Lip amplifier error (falling back):\n{traceback.format_exc()}")
                lp_r[0] = lp_hair
        except Exception as e:
            import traceback
            errs.append(f"LivePortrait: {e}\n{traceback.format_exc()}")

    def run_bd():
        try: bd_r[0]=run_body_animation(source,trimmed,log)
        except Exception as e: errs.append(f"Body: {e}")

    t_lp=threading.Thread(target=run_lp); t_bd=threading.Thread(target=run_bd)
    t_lp.start(); t_bd.start(); t_lp.join(); t_bd.join()
    if errs: raise RuntimeError("\n".join(errs))
    log("✅ Stage 1 complete")
    clear_gpu_memory(log)
    final=composite_videos(lp_r[0],bd_r[0],source,trimmed,log)
    if enable_upscale:
        final=run_esrgan(final,use_dual_gpu=use_dual_gpu,log=log)
    log("━"*52)
    log(f"🎉 Pipeline complete → {final}")
    log("━"*52)
    return final

print("✅ All pipeline functions ready")
print("   LP (GPU 0) ∥ Body (CPU)  →  hair tracker  →  lip amplifier  →  clear VRAM  →  ESRGAN")
print("\n▶ Run Cell 2 to launch Gradio UI")


✅ LivePortrait present
✅ LivePortrait weights present
✅ ESRGAN weights present
✅ All pipeline functions ready
   LP (GPU 0) ∥ Body (CPU)  →  hair tracker  →  lip amplifier  →  clear VRAM  →  ESRGAN

▶ Run Cell 2 to launch Gradio UI


In [17]:
# ── CELL 2: NeuroAnimate — Gradio UI ─────────────────────────────────────────
import gradio as gr, subprocess, os, time, threading

WORK = "/kaggle/working"

def clear_vram_ui():
    r = subprocess.run(["python", "-c", """
import torch
n = torch.cuda.device_count()
lines = []
for i in range(n):
    with torch.cuda.device(i):
        torch.cuda.empty_cache(); torch.cuda.ipc_collect()
    lines.append(f'GPU {i}: {torch.cuda.memory_allocated(i)/1e9:.3f} GB after clear')
print('\\n'.join(lines))
"""], capture_output=True, text=True)
    return "🧹 VRAM Cleared\n" + r.stdout.strip()


def fmt_elapsed(total_secs):
    m = int(total_secs // 60); s = int(total_secs % 60)
    return f"{m}m {s}s" if m > 0 else f"{s}s"


with gr.Blocks(theme=gr.themes.Soft(), title="NeuroAnimate") as demo:

    gr.Markdown("# 🎭 NeuroAnimate: Neural-Driven Portrait Animation")

    with gr.Row():

        # ── LEFT: Settings ────────────────────────────────────────────────
        with gr.Column(scale=1):
            gr.Markdown("### ⚙️ Settings")

            motion_mult = gr.Slider(
                0.3, 1.0, value=0.55, step=0.05,
                label="Motion Multiplier",
                info="0.55 = natural  ·  higher = exaggerated"
            )

            enable_upscale = gr.Checkbox(
                value=True,
                label="Enable 1.5× AI Upscaling (ESRGAN)"
            )

            dual_gpu = gr.Checkbox(
                value=True,
                label="Dual GPU — ESRGAN (GPU 0 ∥ GPU 1)"
            )

            gr.Markdown("### 🖥️ GPU Management")

            with gr.Row():
                clear_btn = gr.Button("🧹 Clear GPU VRAM")

            clear_status = gr.Textbox(
                label="VRAM Status",
                lines=3,
                interactive=False,
                placeholder="VRAM status appears here …"
            )

        # ── RIGHT: Inputs + Output ────────────────────────────────────────
        with gr.Column(scale=2):

            with gr.Row():
                source_img = gr.Image(
                    label="📷 Source Portrait",
                    type="filepath",
                    height=250
                )
                driving_vid = gr.Video(
                    label="🎬 Driving Video",
                    height=250,
                    sources=["upload"]
                )

            with gr.Row():
                run_btn = gr.Button(
                    "🚀 Generate Video",
                    variant="primary"
                )

            video_output = gr.Video(
                label="🎥 Output Video",
                height=400,
                autoplay=True
            )

    # ── Status bar at bottom ──────────────────────────────────────────────
    gen_status = gr.Textbox(
        label="📊 Generation Status",
        value="✅ Ready — Upload a portrait and driving video, then click Generate",
        interactive=False
    )

    # ── Wiring ────────────────────────────────────────────────────────────
    def generate_with_state(source_image, driving_video, motion_mult, enable_upscale, dual_gpu):
        if source_image is None or driving_video is None:
            yield None, "⚠️ Please upload both a portrait image and a driving video."
            return

        result_h = [None]; error_h = [None]; logs = []
        def log(msg): logs.append(str(msg))

        def run():
            try:
                result_h[0] = full_pipeline(
                    source_image_path=source_image,
                    driving_video_path=driving_video,
                    motion_multiplier=float(motion_mult),
                    enable_upscale=enable_upscale,
                    use_dual_gpu=dual_gpu,
                    log=log,
                )
            except Exception as e:
                import traceback
                error_h[0] = f"{e}\n{traceback.format_exc()}"

        t = threading.Thread(target=run)
        t.start()
        start_ts = time.time()
        yield None, "⏱  Generating…  0s elapsed"

        while t.is_alive():
            time.sleep(1.5)
            elapsed = fmt_elapsed(time.time() - start_ts)
            yield None, f"⏱  Generating…  {elapsed} elapsed"

        t.join()
        total = fmt_elapsed(time.time() - start_ts)

        if error_h[0]:
            yield None, f"❌ Error after {total}\n{error_h[0][:200]}"
        else:
            yield result_h[0], f"✅ Video Generated Successfully  ·  Total time: {total}"

    run_btn.click(
        fn=generate_with_state,
        inputs=[source_img, driving_vid, motion_mult, enable_upscale, dual_gpu],
        outputs=[video_output, gen_status],
    )

    clear_btn.click(
        fn=clear_vram_ui,
        inputs=[],
        outputs=[clear_status]
    )

demo.launch(share=True, debug=False)


/tmp/ipykernel_55/259191743.py:25: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="NeuroAnimate") as demo:


* Running on local URL:  http://127.0.0.1:7865
* Running on public URL: https://2470d74689d2d086c7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
